I am now Going to try rewriting my 2 parrallel interface code with matrice's. This will make it far simpler to scale to n number of interfaces. I am going to make a general transmission matrix D and a general propagation matrix P. These matrixes will be functions of our fresnell coeeficients. Then I am Going to Make our transmission matrix M comprised our our string of D and P matrice's evaluatued at different interfaces. Once I have matrix M i can solve for t and p total. 

In [8]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd    

First I have put our plotting function here. In the next iteration I want to change d to d1 to make it extend with to a case multiple d's but I am leaving for now. 

In [ ]:
def totalFresnalcoeffPlot(func1, func2, func3, func4):
    fig0 = plt.figure(figsize=(8,5))
    ax0 = fig0.add_subplot(111)
    th0 = np.linspace(0.1,89.9,10000)
    line01, = ax0.plot([],[], 'r--')
    line02, = ax0.plot([],[], 'b--')
    line03, = ax0.plot([],[], 'r-')
    line04, = ax0.plot([],[], 'b-')
    ax0.set_xlim((0,90))
    ax0.set_ylim((-1,1))
    ax0.axhline(0, linewidth=0.75, color='k')
    # ax0.axvline(0, color='k')
    ax0.set_xlabel(r'$\theta_i$')
    ax0.set_ylabel('amplitude coefficient')
    ax0.legend([r'rperptotal, r$_s$', r'tperptotal, t$_s$', r'rparatotal, r$_p$', r'tparatotal, t$_p$'],)
    def plot(n0=1, n1=1.4, n2=1.5, k0=0, k1=0, k2=0, d=1, Wavelength=4):
        output1 = func1(n0, n1, n2, k0, k1, k2, d, th0, Wavelength)
        output2 = func2(n0, n1, n2, k0, k1, k2, d, th0, Wavelength)
        output3 = func3(n0, n1, n2, k0, k1, k2, d, th0, Wavelength)
        output4 = func4(n0, n1, n2, k0, k1, k2, d, th0, Wavelength)
        line01.set_data(th0, output1)
        line02.set_data(th0, output2)
        line03.set_data(th0, output3)
        line04.set_data(th0, output4)
        plt.draw()
    interact(plot, n0=(0,5,.1), n1=(0,5,.1), n2=(0,5,.1), k0=(0,5,.1), k1=(0,5,.1), k2=(0,5,.1), d=(0,5,.1), Wavelength=(0,5,.1)) 


Next I am putting in some general functiions that can be used with specific inputs later on. 

In [ ]:
def cos(th):
    return np.cos(th*np.pi/180) 

def costht(ni, nt, ki, kt, thi):
    return(np.sqrt(1-((ni+1j*ki)/(nt+1j*kt)*sin(thi))**2)) 

def arcsin(ratio):
    return np.arcsin(ratio)*180/np.pi 

def sin(th):
    return np.sin(th*np.pi/180) 

# 3 Returns transmitted angle from incident angle
def snells(ni, nt, ki, kt, thi):
    return(arcsin((ni+1j*ki)/(nt+1j*kt)*sin(thi)))

# 4 General form of fresnel coefficients for s polarized light (perpindicular electric field) 
def r_s(ni, nt, ki, kt, thi, mui=8.85*10**-12, mut=8.85*10**-12):
    cos_thi = cos(thi)
    cos_tht = costht(ni, nt, ki, kt, thi)
    return(((ni+1j*ki)/mui*cos_thi-(nt+1j*kt)/mut*cos_tht)/((ni+1j*ki)/mui*cos_thi+(nt+1j*kt)/mut*cos_tht)) 

def t_s(ni, nt, ki, kt, thi, mui=8.85*10**-12, mut=8.85*10**-12):
    cos_thi = cos(thi)
    cos_tht = costht(ni, nt, ki, kt, thi)  
    return((2*(ni+1j*ki)/mui*cos_thi)/((ni+1j*ki)/mui*cos_thi+(nt+1j*kt)/mut*cos_tht))

# 5 General form of fresnel coefficients for p polarized light (parrallel electric field)  

def r_p(ni, nt, ki, kt, thi, mui=8.85*10**-12, mut=8.85*10**-12):
    cos_thi = cos(thi)
    cos_tht = costht(ni, nt, ki, kt, thi)
    
    return((-(nt+1j*kt)/mut*cos_thi+(ni+1j*ki)/mui*cos_tht)/((nt+1j*kt)/mut*cos_thi+(ni+1j*ki)/mui*cos_tht))

def t_p(ni, nt, ki, kt, thi, mui=8.85*10**-12, mut=8.85*10**-12):
    cos_thi = cos(thi)
    cos_tht = costht(ni, nt, ki, kt, thi)
    
    return((2*(ni+1j*ki)/mui*cos_thi)/((nt+1j*kt)/mut*cos_thi+(ni+1j*ki)/mui*cos_tht))  


Now I will create our transmission matrix D. D will be a function of any t and r. This means that d needs to be a function of ni, nt, ki, kt, thi, mui=8.85*10**-12, mut=8.85*10**-12). I think I can have d in terms of r and p and send multiple functions through it but we will see. 

In [16]:
def D_p ():

    
    return(1/t * np.array([[1,r],[r,1]]))


now to make my propagation matrix p. 

In [19]:
def P_p (ni, nt, ki, kt, thi, d, Wavelength ):

    cos_tht = costht(ni, nt, ki, kt, thi)
    return( np.array([[np.exp(1j*2*np.pi*d*(nt+1j*kt)*cos_tht/Wavelength),0],[0,np.exp(-1j*2*np.pi*d*(nt+1j*kt)*cos_tht/Wavelength)]]))


now for transfer matrix M

In [ ]:
def M_p ():
    
    return(D_p (n0, n1, k0, k1, th0, mui=8.85*10**-12, mut=8.85*10**-12) @ p() * D_p (n1, n2, k1, k2, th1, mui=8.85*10**-12, mut=8.85*10**-12))

In [ ]:
def t_p_tot(n0, n1, n2, k0, k1, k2, d, th0, Wavelength):
    th1 = snells(n0, n1, k0, k1, th0)
    cos_th1 = cos(th1)
    t_p_nin0_ntn1 = t_p(n0, n1, k0, k1, th0, mui=8.85*10**-12, mut=8.85*10**-12)
    t_p_nin1_ntn2 = t_p(n1, n2, k1, k2, th1, mui=8.85*10**-12, mut=8.85*10**-12)
    r_p_nin1_ntn0 = r_p(n1, n0, k1, k0, th1, mui=8.85*10**-12, mut=8.85*10**-12) 
    r_p_nin1_ntn2 = r_p(n1, n2, k1, k2, th1, mui=8.85*10**-12, mut=8.85*10**-12)
    return((t_p_nin0_ntn1*np.exp(1j*2*np.pi*d*(n1+1j*k1)*cos_th1/Wavelength)*t_p_nin1_ntn2)/(1-r_p_nin1_ntn0*np.exp(2*1j*2*np.pi*d*(n1+1j*k1)*cos_th1/Wavelength)*r_p_nin1_ntn2))
